# 🏎️ Step 07 — Formula 1 WDC 2026 Prediction

**Project:** Formula 1 World Drivers' Championship 2026 Analysis

This notebook builds a data-driven **WDC prediction model** using the driver performance features produced in Notebook 05.

> The prediction is an analytical portfolio result, not an official Formula 1 forecast. It uses the completed races currently represented in `driver_features.csv`.

### Pipeline
`01 Data Collection → 02 Collect All Results → 03 Data Cleaning → 04 EDA → 05 Feature Engineering → 06 Dashboard → 07 Prediction`


## Notebook Dependency

Run the notebooks in this order:

`01_data_collection` → `02_collect_all_races` → `03_data_cleaning` → `04_exploratory_data_analysis` → `05_feature_engineering` → `06_dashboard` → `07_prediction`

For the final portfolio flow, **run 07_prediction before rerunning the prediction-integration cell in 06_dashboard**, because Notebook 06 reads `wdc_2026_prediction.csv` produced by Notebook 07.

## Step 1 — Import Libraries

### Objective 🇬🇧
Import the libraries required for data preparation, machine learning, evaluation, and visualization.

### Tujuan 🇮🇩
Mengimpor library yang diperlukan untuk persiapan data, machine learning, evaluasi, dan visualisasi.

### Expected Output
All required libraries are imported successfully.

In [1]:
# ==========================================
# Step 1 — Import Libraries
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Libraries imported successfully!")


✅ Libraries imported successfully!


## Step 2 — Load Driver Features

### Objective 🇬🇧
Load the driver-level feature dataset generated by Notebook 05.

### Tujuan 🇮🇩
Memuat dataset fitur pembalap yang dihasilkan oleh Notebook 05.

### Expected Output
The dataset is loaded successfully.

In [2]:
from pathlib import Path

# Robust project-relative path
project_path = Path("..")
processed_path = project_path / "data" / "processed"
dataset_path = processed_path / "driver_features.csv"

if not dataset_path.exists():
    raise FileNotFoundError(f"Driver feature file not found: {dataset_path.resolve()}")

driver_features = pd.read_csv(dataset_path)

print("✅ Driver feature dataset loaded successfully!")
print(f"Dataset shape: {driver_features.shape}")
display(driver_features.head())

✅ Driver feature dataset loaded successfully!
Dataset shape: (23, 15)


,FullName,Team,TotalRaces,TotalPoints,Wins,Podiums,AvgFinish,AvgGrid,WinRate,PodiumRate,PointsPerRace,BestFinish,WorstFinish,AveragePositionGain,DNFRate
0,Alexander Albon,Williams,11,5.0,0,0,16.363636,16.545455,0.00,0.00,0.45,8.0,22.0,0.181818,72.73
1,Arvid Lindblad,Racing Bulls,11,20.0,0,0,11.363636,10.818182,0.00,0.00,1.82,7.0,22.0,-0.545455,63.64
2,Carlos Sainz,Williams,11,6.0,0,0,14.545455,16.272727,0.00,0.00,0.55,9.0,20.0,1.727273,72.73
3,Charles Leclerc,Ferrari,11,127.0,1,4,5.363636,3.909091,9.09,36.36,11.55,1.0,17.0,-1.454545,9.09
4,Esteban Ocon,Haas F1 Team,11,3.0,0,0,13.818182,14.909091,0.00,0.00,0.27,9.0,19.0,1.090909,63.64


### Interpretation 🇬🇧
Each row represents a driver and the columns summarize performance across the races represented in the dataset.

### Interpretasi 🇮🇩
Setiap baris merepresentasikan seorang pembalap dan kolom-kolomnya merangkum performa selama race yang tersedia dalam dataset.

In [3]:
print("Number of drivers:", driver_features["FullName"].nunique())
print("Maximum races represented:", driver_features["TotalRaces"].max())
print("Race-count distribution:")
display(driver_features["TotalRaces"].value_counts().sort_index())

print("\nTotal points range:",
      driver_features["TotalPoints"].min(),
      "to",
      driver_features["TotalPoints"].max())

Number of drivers: 23
Maximum races represented: 11
Race-count distribution:


TotalRaces
1      1
10     1
11    21
Name: count, dtype: int64


Total points range: 0.0 to 216.0


## Step 3 — Validate Prediction Dataset

### Objective 🇬🇧
Check missing values, duplicate drivers, data types, and race coverage before modelling.

### Tujuan 🇮🇩
Memeriksa missing values, duplikasi pembalap, tipe data, dan cakupan race sebelum modelling.

In [4]:
print("=== Missing Values ===")
display(driver_features.isnull().sum().to_frame("Missing Values"))

print("\n=== Duplicate Drivers ===")
print("Duplicate driver rows:", driver_features["FullName"].duplicated().sum())

print("\n=== Race Coverage ===")
display(driver_features["TotalRaces"].value_counts().sort_index())

print("\n=== Data Types ===")
display(driver_features.dtypes.to_frame("Data Type"))

print("\n⚠️ Note: driver_features reflects the cleaned data available from the upstream notebooks.")

=== Missing Values ===


,Missing Values
FullName,0
Team,0
TotalRaces,0
TotalPoints,0
Wins,0
Podiums,0
AvgFinish,0
AvgGrid,0
WinRate,0
PodiumRate,0



=== Duplicate Drivers ===
Duplicate driver rows: 0

=== Race Coverage ===


TotalRaces
1      1
10     1
11    21
Name: count, dtype: int64


=== Data Types ===


,Data Type
FullName,object
Team,object
TotalRaces,int64
TotalPoints,float64
Wins,int64
Podiums,int64
AvgFinish,float64
AvgGrid,float64
WinRate,float64
PodiumRate,float64



⚠️ Note: driver_features reflects the cleaned data available from the upstream notebooks.


In [5]:
# Remove accidental duplicate driver records, if any
driver_features = driver_features.drop_duplicates(subset=["FullName"]).copy()

print("✅ Validation completed.")
print("Drivers after validation:", len(driver_features))


✅ Validation completed.
Drivers after validation: 23


## Step 4 — Define Prediction Features

### Objective 🇬🇧
Select performance indicators representing championship competitiveness.

### Tujuan 🇮🇩
Memilih indikator performa yang merepresentasikan kekuatan pembalap dalam championship.

`FullName` and `Team` are identifiers, so they are not used directly as numerical model inputs.

In [6]:
# ==========================================
# Step 4 — Define Prediction Features
# ==========================================

feature_columns = [
    "TotalPoints",
    "Wins",
    "Podiums",
    "AvgFinish",
    "AvgGrid",
    "WinRate",
    "PodiumRate",
    "PointsPerRace",
    "BestFinish",
    "AveragePositionGain",
    "DNFRate",
]

X = driver_features[feature_columns].copy()
y = driver_features["TotalPoints"].copy()

# Convert lower-is-better metrics into positive-performance direction
X["AvgFinish"] = -X["AvgFinish"]
X["AvgGrid"] = -X["AvgGrid"]
X["BestFinish"] = -X["BestFinish"]
X["DNFRate"] = -X["DNFRate"]

print("Selected features:")
print(feature_columns)

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)


Selected features:
['TotalPoints', 'Wins', 'Podiums', 'AvgFinish', 'AvgGrid', 'WinRate', 'PodiumRate', 'PointsPerRace', 'BestFinish', 'AveragePositionGain', 'DNFRate']

Feature matrix shape: (23, 11)
Target shape: (23,)


## Step 5 — Standardize Features

### Objective 🇬🇧
Standardize numerical variables with different scales.

### Tujuan 🇮🇩
Melakukan standardisasi pada variabel numerik yang memiliki skala berbeda.

In [7]:
# ==========================================
# Step 5 — Feature Scaling
# ==========================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=feature_columns,
    index=driver_features.index
)

print("✅ Features standardized successfully!")
display(X_scaled.head())


✅ Features standardized successfully!


,TotalPoints,Wins,Podiums,AvgFinish,AvgGrid,WinRate,PodiumRate,PointsPerRace,BestFinish,AveragePositionGain,DNFRate
0,-0.739695,-0.363848,-0.582272,-1.161515,-0.897999,-0.363828,-0.582252,-0.744261,-0.434063,0.068654,-0.624365
1,-0.483475,-0.363848,-0.582272,0.025716,0.123833,-0.363828,-0.582252,-0.487172,-0.190564,-0.284206,-0.316925
2,-0.722614,-0.363848,-0.582272,-0.729795,-0.849340,-0.363828,-0.582252,-0.725495,-0.677562,0.818482,-0.624365
3,1.344225,0.396925,1.041032,1.450392,1.356519,0.396828,1.040943,1.338723,1.270428,-0.725281,1.528054
4,-0.773858,-0.363848,-0.582272,-0.557107,-0.606047,-0.363828,-0.582252,-0.778039,-0.677562,0.509729,-0.316925


## Step 6 — Build Prediction Model

### Objective 🇬🇧
Train a Random Forest regression model to learn the relationship between driver performance features and championship points.

### Tujuan 🇮🇩
Melatih Random Forest Regression untuk mempelajari hubungan antara fitur performa pembalap dan championship points.

### Methodological Note
Because the dataset contains one aggregated row per driver, this is best presented as an **analytical scoring/prediction model**, not as a definitive statistical forecast of the final WDC.

In [8]:
# ==========================================
# Step 6 — Train Random Forest Regression
# ==========================================

model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    max_depth=4,
    min_samples_leaf=2,
    n_jobs=-1
)

model.fit(X_scaled, y)

print("✅ Random Forest model trained successfully!")


✅ Random Forest model trained successfully!


## Step 7 — Model Validation

### Objective 🇬🇧
Estimate model performance using Leave-One-Out Cross-Validation (LOOCV), which is suitable for a small driver-level dataset.

### Tujuan 🇮🇩
Mengestimasi performa model menggunakan LOOCV karena jumlah observasi pada level pembalap relatif kecil.

### Metrics
- MAE
- RMSE
- R²

These metrics should be interpreted as diagnostics of how well the model reconstructs the current championship-points pattern.

In [9]:
# ==========================================
# Step 7 — LOOCV Validation
# ==========================================

loo = LeaveOneOut()

mae_scores = -cross_val_score(
    model, X_scaled, y, cv=loo,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

mse_scores = -cross_val_score(
    model, X_scaled, y, cv=loo,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

r2_scores = cross_val_score(
    model, X_scaled, y, cv=loo,
    scoring="r2",
    n_jobs=-1
)

print("=== LOOCV Results ===")
print(f"MAE : {mae_scores.mean():.2f}")
print(f"RMSE: {np.sqrt(mse_scores.mean()):.2f}")
print(f"R²  : {r2_scores.mean():.3f}")


=== LOOCV Results ===
MAE : 7.49
RMSE: 19.90
R²  : nan


### Interpretation 🇬🇧
LOOCV is used because there are relatively few driver-level observations. The scores should not be interpreted as proof that the model can predict the final official WDC with the same accuracy.

### Interpretasi 🇮🇩
LOOCV digunakan karena jumlah observasi pembalap relatif sedikit. Nilai evaluasi tidak boleh dianggap sebagai bukti bahwa model akan memprediksi hasil akhir WDC dengan tingkat akurasi yang sama.

## Step 8 — Generate Predicted Championship Score

### Objective 🇬🇧
Generate a model-based predicted championship score for every driver.

### Tujuan 🇮🇩
Menghasilkan predicted championship score untuk setiap pembalap berdasarkan model.

In [10]:
# ==========================================
# Step 8 — Generate Predictions
# ==========================================

driver_features["PredictedPoints"] = model.predict(X_scaled)
driver_features["PredictedPoints"] = driver_features["PredictedPoints"].clip(lower=0)

prediction = (
    driver_features[
        ["FullName", "Team", "TotalRaces", "TotalPoints", "PredictedPoints"]
    ]
    .sort_values("PredictedPoints", ascending=False)
    .reset_index(drop=True)
)

prediction["PredictedRank"] = np.arange(1, len(prediction) + 1)

prediction["PredictedPoints"] = prediction["PredictedPoints"].round(2)

prediction = prediction[
    [
        "PredictedRank",
        "FullName",
        "Team",
        "TotalRaces",
        "TotalPoints",
        "PredictedPoints",
    ]
]

print("✅ Championship predictions generated!")
display(prediction)


✅ Championship predictions generated!


,PredictedRank,FullName,Team,TotalRaces,TotalPoints,PredictedPoints
0,1,Kimi Antonelli,Mercedes,11,216.0,155.29
1,2,Lewis Hamilton,Ferrari,11,138.0,142.32
2,3,George Russell,Mercedes,11,131.0,138.81
3,4,Charles Leclerc,Ferrari,11,127.0,135.36
4,5,Lando Norris,McLaren,11,112.0,117.11
5,6,Max Verstappen,Red Bull Racing,11,88.0,92.98
6,7,Oscar Piastri,McLaren,11,73.0,76.11
7,8,Isack Hadjar,Red Bull Racing,10,60.0,57.46
8,9,Liam Lawson,Red Bull Racing,11,42.0,43.28
9,10,Pierre Gasly,Alpine,11,36.0,40.33


## Step 9 — Championship Probability

### Objective 🇬🇧
Convert predicted scores into a relative championship probability for easier comparison.

### Tujuan 🇮🇩
Mengubah predicted score menjadi probabilitas relatif agar hasil lebih mudah dibandingkan.

> This is a **model-derived relative probability**, not an official Formula 1 probability.

In [11]:
# ==========================================
# Step 9 — Relative Championship Probability
# ==========================================

prediction["ChampionshipProbability"] = (
    prediction["PredictedPoints"]
    / prediction["PredictedPoints"].sum()
    * 100
).round(2)

display(prediction)


,PredictedRank,FullName,Team,TotalRaces,TotalPoints,PredictedPoints,ChampionshipProbability
0,1,Kimi Antonelli,Mercedes,11,216.0,155.29,14.15
1,2,Lewis Hamilton,Ferrari,11,138.0,142.32,12.97
2,3,George Russell,Mercedes,11,131.0,138.81,12.65
3,4,Charles Leclerc,Ferrari,11,127.0,135.36,12.33
4,5,Lando Norris,McLaren,11,112.0,117.11,10.67
5,6,Max Verstappen,Red Bull Racing,11,88.0,92.98,8.47
6,7,Oscar Piastri,McLaren,11,73.0,76.11,6.93
7,8,Isack Hadjar,Red Bull Racing,10,60.0,57.46,5.24
8,9,Liam Lawson,Red Bull Racing,11,42.0,43.28,3.94
9,10,Pierre Gasly,Alpine,11,36.0,40.33,3.67


## Step 10 — Predicted WDC Champion

### Objective 🇬🇧
Identify the driver with the highest model-based predicted championship score.

### Tujuan 🇮🇩
Menentukan pembalap dengan predicted championship score tertinggi.

In [12]:
# ==========================================
# Step 10 — Predicted WDC Champion
# ==========================================

predicted_champion = prediction.iloc[0]

print("🏆 PREDICTED 2026 WDC CHAMPION")
print("=" * 45)
print(f"Driver: {predicted_champion['FullName']}")
print(f"Team: {predicted_champion['Team']}")
print(f"Predicted Points: {predicted_champion['PredictedPoints']:.2f}")
print(f"Relative Probability: {predicted_champion['ChampionshipProbability']:.2f}%")


🏆 PREDICTED 2026 WDC CHAMPION
Driver: Kimi Antonelli
Team: Mercedes
Predicted Points: 155.29
Relative Probability: 14.15%


## Step 11 — Predicted Championship Ranking

### Objective 🇬🇧
Visualize the predicted WDC ranking.

### Tujuan 🇮🇩
Memvisualisasikan ranking WDC berdasarkan hasil prediksi.

In [13]:
# ==========================================
# Step 11 — Predicted WDC Ranking
# ==========================================

ranking_fig = px.bar(
    prediction.sort_values("PredictedPoints", ascending=True),
    x="PredictedPoints",
    y="FullName",
    orientation="h",
    color="PredictedPoints",
    title="Predicted 2026 WDC Championship Ranking",
    text="PredictedPoints",
)

ranking_fig.update_layout(
    xaxis_title="Predicted Championship Points",
    yaxis_title="Driver",
    showlegend=False,
)

ranking_fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
)

ranking_fig.show()


## Step 12 — Championship Probability Visualization

### Objective 🇬🇧
Compare relative championship probabilities between drivers.

### Tujuan 🇮🇩
Membandingkan probabilitas relatif championship antar pembalap.

In [14]:
# ==========================================
# Step 12 — Championship Probability
# ==========================================

probability_fig = px.bar(
    prediction.sort_values("ChampionshipProbability", ascending=False),
    x="FullName",
    y="ChampionshipProbability",
    color="ChampionshipProbability",
    title="Relative 2026 WDC Championship Probability",
    text="ChampionshipProbability",
)

probability_fig.update_layout(
    xaxis_title="Driver",
    yaxis_title="Relative Championship Probability (%)",
    showlegend=False,
)

probability_fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
)

probability_fig.show()


## Step 13 — Feature Importance

### Objective 🇬🇧
Identify which performance indicators contribute most to the Random Forest model.

### Tujuan 🇮🇩
Mengidentifikasi indikator performa yang paling berpengaruh terhadap model Random Forest.

In [15]:
# ==========================================
# Step 13 — Feature Importance
# ==========================================

feature_importance = (
    pd.DataFrame({
        "Feature": feature_columns,
        "Importance": model.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
)

display(feature_importance)

importance_fig = px.bar(
    feature_importance.sort_values("Importance", ascending=True),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Random Forest Feature Importance",
    text="Importance",
)

importance_fig.update_layout(
    xaxis_title="Importance",
    yaxis_title="Feature",
    showlegend=False,
)

importance_fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
)

importance_fig.show()


,Feature,Importance
7,PointsPerRace,0.168598
0,TotalPoints,0.146491
4,AvgGrid,0.131827
2,Podiums,0.130301
8,BestFinish,0.104463
10,DNFRate,0.103755
6,PodiumRate,0.087518
3,AvgFinish,0.080845
9,AveragePositionGain,0.020632
5,WinRate,0.014965


## Step 14 — Top 5 WDC Prediction

### Objective 🇬🇧
Present the five highest-ranked drivers according to the model.

### Tujuan 🇮🇩
Menampilkan lima pembalap dengan ranking prediksi tertinggi.

In [16]:
# ==========================================
# Step 14 — Top 5 Prediction
# ==========================================

top_5 = prediction.head(5).copy()

print("🏆 TOP 5 PREDICTED 2026 WDC")
display(top_5)


🏆 TOP 5 PREDICTED 2026 WDC


,PredictedRank,FullName,Team,TotalRaces,TotalPoints,PredictedPoints,ChampionshipProbability
0,1,Kimi Antonelli,Mercedes,11,216.0,155.29,14.15
1,2,Lewis Hamilton,Ferrari,11,138.0,142.32,12.97
2,3,George Russell,Mercedes,11,131.0,138.81,12.65
3,4,Charles Leclerc,Ferrari,11,127.0,135.36,12.33
4,5,Lando Norris,McLaren,11,112.0,117.11,10.67


## Step 15 — Export Prediction Results

### Objective 🇬🇧
Save the prediction results for portfolio documentation and future dashboard integration.

### Tujuan 🇮🇩
Menyimpan hasil prediksi untuk dokumentasi portfolio dan integrasi dashboard di masa depan.

In [17]:
output_path = processed_path / "wdc_2026_prediction.csv"

prediction.to_csv(output_path, index=False)

print("✅ Prediction results exported successfully!")
print(f"📁 Saved to: {output_path.resolve()}")

✅ Prediction results exported successfully!
📁 Saved to: C:\Users\firdhan\Desktop\INTERN BISMILLAH\portofolio\Analisis F1 WDC 2026\data\processed\wdc_2026_prediction.csv


# 🏁 Final Interpretation

### English

The model ranks Formula 1 drivers according to their current performance profile across the completed races represented in the dataset. It combines championship points, wins, podiums, finishing performance, qualifying performance, points per race, and DNF-related indicators.

The driver ranked first is presented as the model's **current predicted WDC champion**.

This is an **analytical portfolio prediction**, not an official Formula 1 forecast. When another Grand Prix is completed, rerun Notebooks 02–05 and this notebook to update the result.

### Bahasa Indonesia

Model memberikan ranking pembalap Formula 1 berdasarkan profil performa pada race yang telah selesai dan tersedia di dataset. Model menggunakan championship points, kemenangan, podium, performa finishing, qualifying, points per race, dan indikator DNF.

Pembalap dengan ranking pertama ditampilkan sebagai **prediksi WDC champion saat ini**.

Hasil ini merupakan **prediksi analitis untuk portfolio**, bukan prediksi resmi Formula 1. Setelah Grand Prix baru selesai, Notebook 02–05 dan notebook ini dapat dijalankan kembali untuk memperbarui hasil.

In [18]:
print("🏎️ FORMULA 1 WDC 2026 — PREDICTION SUMMARY")
print("=" * 60)

print(f"Drivers analyzed: {len(prediction)}")
print(f"Maximum completed races represented: {driver_features['TotalRaces'].max()}")

print(f"\n🏆 Predicted WDC Champion: {prediction.iloc[0]['FullName']}")
print(f"Team: {prediction.iloc[0]['Team']}")
print(f"Predicted Points: {prediction.iloc[0]['PredictedPoints']:.2f}")
print(f"Relative Probability: {prediction.iloc[0]['ChampionshipProbability']:.2f}%")

print("\n🏁 Top 5:")
for _, row in prediction.head(5).iterrows():
    print(
        f"{int(row['PredictedRank'])}. "
        f"{row['FullName']} — "
        f"{row['PredictedPoints']:.2f} predicted points — "
        f"{row['ChampionshipProbability']:.2f}% relative probability"
    )

print("\n⚠️ Relative probability is a model-derived comparison, not an official F1 probability.")
print("✅ Prediction notebook completed successfully!")

🏎️ FORMULA 1 WDC 2026 — PREDICTION SUMMARY
Drivers analyzed: 23
Maximum completed races represented: 11

🏆 Predicted WDC Champion: Kimi Antonelli
Team: Mercedes
Predicted Points: 155.29
Relative Probability: 14.15%

🏁 Top 5:
1. Kimi Antonelli — 155.29 predicted points — 14.15% relative probability
2. Lewis Hamilton — 142.32 predicted points — 12.97% relative probability
3. George Russell — 138.81 predicted points — 12.65% relative probability
4. Charles Leclerc — 135.36 predicted points — 12.33% relative probability
5. Lando Norris — 117.11 predicted points — 10.67% relative probability

⚠️ Relative probability is a model-derived comparison, not an official F1 probability.
✅ Prediction notebook completed successfully!
